# TCP Agent and Collector

The previous notebook prepared profiling data for transport:

```text
ProfileEvent
    ↓
serialize()
    ↓
payload bytes
    ↓
length-prefix framing
    ↓
framed event
    ↓
batching
    ↓
batch buffer
```

At that point, all work still happened inside one process.

The goal of this notebook is to move from:

```text
local in-process data handling
```

to:

```text
inter-process profiling data transport
```

using a TCP client-server architecture.

---

## Overall Architecture

The system will contain two main sides:

```text
Profiler Agent
```

and:

```text
Collector
```

The profiler agent produces profiling data and sends it over TCP.

The collector receives the byte stream, reconstructs the messages, and converts them back into `ProfileEvent` objects.

The complete path is:

```text
Application / Profiler
        ↓
   ProfileEvent
        ↓
     Serializer
        ↓
  Serialized Payload
        ↓
    FrameCodec
        ↓
 [length][payload]
        ↓
   BatchBuilder
        ↓
    Batch Buffer
        ↓
   TCP Client
        ↓
========================
       TCP Stream
========================
        ↓
   TCP Server
        ↓
 Receive Buffer
        ↓
   BatchParser
        ↓
 Individual Frames
        ↓
    FrameCodec
        ↓
 Serialized Payload
        ↓
   Deserializer
        ↓
   ProfileEvent
        ↓
     Collector
```

---

## Agent Side

The agent acts as the producer of profiling data.

Its responsibility is:

```text
generate profiling events
        ↓
serialize them
        ↓
frame them
        ↓
batch them
        ↓
send them over TCP
```

Conceptually:

```text
ProfileEvent
    ↓
SimpleTextSerializer
    ↓
FrameCodec
    ↓
BatchBuilder
    ↓
TCP send()
```

The agent should not need to know how the collector stores or analyzes the events.

Its responsibility is only:

```text
produce + encode + transmit
```

---

## Collector Side

The collector acts as the receiver.

Its responsibility is:

```text
receive TCP bytes
        ↓
reconstruct complete messages
        ↓
parse frames
        ↓
deserialize events
        ↓
process profiling data
```

Conceptually:

```text
TCP recv()
    ↓
receive buffer
    ↓
BatchParser
    ↓
FrameCodec
    ↓
SimpleTextDeserializer
    ↓
ProfileEvent
```

Later, the collector can send reconstructed events to:

```text
ConsoleSink
database
file storage
performance analyzer
```

---

## Why TCP?

TCP provides:

```text
reliable
ordered
byte-stream communication
```

This is useful for profiling infrastructure because the receiver should observe profiling data in the same byte order in which it was sent.

However, TCP does not preserve application-level message boundaries.

For example, if the sender performs:

```cpp
send(message1);
send(message2);
```

the receiver may observe:

```text
recv 1 → part of message1

recv 2 → rest of message1 + part of message2

recv 3 → rest of message2
```

Therefore:

```text
send() boundary
```

does not equal:

```text
recv() boundary
```

This is one of the most important concepts in this notebook.

---

## TCP Is a Byte Stream

The network should be viewed as:

```text
byte byte byte byte byte byte byte ...
```

rather than:

```text
message 1
message 2
message 3
```

That is why the framing work from the previous notebook is necessary.

Each event already has:

```text
[length][payload]
```

so the receiver can reconstruct application-level messages from the TCP byte stream.

---

## Connection Lifecycle

The collector will act as the server.

Its lifecycle is:

```text
socket()
    ↓
bind()
    ↓
listen()
    ↓
accept()
    ↓
recv()
    ↓
process data
    ↓
close()
```

The agent will act as the client.

Its lifecycle is:

```text
socket()
    ↓
connect()
    ↓
send()
    ↓
close()
```

Together:

```text
Collector                        Agent

socket()                         socket()
   ↓                                ↓
bind()                           connect()
   ↓                                │
listen()                            │
   ↓                                │
accept()  ←─────────────────────────┘
   │
   │          TCP connection
   │<──────────────────────────────>│
   │                                │
recv()  ←──────────────────────  send()
```

---

## Partial `send()` and `recv()`

Another major concept is that:

```cpp
send(...)
```

is not guaranteed to transmit the entire requested buffer in one call.

Similarly:

```cpp
recv(...)
```

is not guaranteed to return one complete message.

Therefore, production-style networking code needs helper functions such as:

```text
send_all()
```

and:

```text
recv_exact()
```

Conceptually:

```text
send_all(buffer)
    ↓
send remaining bytes
    ↓
partial send?
    ├── yes → continue
    └── no  → finished
```

and:

```text
recv_exact(N)
    ↓
receive bytes
    ↓
received N bytes?
    ├── no  → continue
    └── yes → finished
```

---

## Why Batching Matters More With TCP

In the previous notebook, batching mostly reduced logical batch count.

With TCP, batching becomes more important because each network operation has fixed overhead.

Without batching:

```text
event 1 → send()
event 2 → send()
event 3 → send()
event 4 → send()
```

With batching:

```text
event 1 ┐
event 2 ├→ one batch → send()
event 3 │
event 4 ┘
```

This can reduce:

```text
number of send() calls
system-call overhead
network-stack overhead
per-message processing overhead
```

Therefore, the throughput effect of batching should become more visible in this notebook.

---

## Failure Conditions

A real client-server pipeline must also consider failure cases.

Examples include:

```text
client disconnects
server closes connection
recv() returns 0
partial message arrives
invalid frame length
malformed serialized event
send() fails
recv() fails
```

The goal is not to build a complete production networking library.

The goal is to understand how a profiling transport pipeline should detect and handle these conditions safely.

---

## Notebook Roadmap

This notebook will proceed through the following stages:

```text
Standalone Setup
        ↓
Basic TCP Client / Server
        ↓
Send One Profiling Event
        ↓
Partial send() / recv() Handling
        ↓
Send Batched Events
        ↓
Collector Reconstruction
        ↓
Connection and Failure Handling
        ↓
TCP Performance Benchmark
```

---

## Final Target

By the end of this notebook, the system should demonstrate:

```text
Profiler Agent
    ↓
ProfileEvent
    ↓
serialize
    ↓
frame
    ↓
batch
    ↓
TCP
    ↓
Collector
    ↓
parse
    ↓
deserialize
    ↓
ProfileEvent
```

This creates the first real distributed version of the profiling pipeline.

---

## Key Questions

By the end of this notebook, the following questions should be answerable:

- What is the difference between a TCP client and a TCP server?
- What do `socket()`, `bind()`, `listen()`, `accept()`, and `connect()` do?
- Why is TCP described as a byte stream?
- Why does one `send()` not necessarily correspond to one `recv()`?
- What is a partial send?
- What is a partial receive?
- Why is message framing required on top of TCP?
- How does length-prefix framing help reconstruct messages?
- Why can batching reduce network overhead?
- How should the collector reconstruct `ProfileEvent` objects?
- What happens when a client disconnects?
- How would this architecture scale from one profiling agent to multiple agents?

## Step 1 — Standalone Setup

This notebook is designed to run independently in a fresh Google Colab runtime.

The setup installs and recreates the minimal C++20 project components required for the TCP agent-and-collector pipeline.

Required tools:

- GCC / G++ with C++20 support
- CMake
- Ninja
- Git
- POSIX socket support
- POSIX thread support

This notebook does not require a GPU.

The focus is on host-side distributed systems programming:

- TCP client-server communication
- socket lifecycle management
- sending and receiving profiling data
- partial `send()` and `recv()` handling
- message framing over a byte stream
- batched event transport
- collector-side reconstruction
- connection shutdown and error handling
- network throughput and latency measurement

The notebook reuses the data-format components developed previously:

```text
ProfileEvent
    ↓
SimpleTextSerializer
    ↓
FrameCodec
    ↓
BatchBuilder
```

and extends the pipeline with networking:

```text
Profiler Agent
    ↓
TCP Client
    ↓
TCP Stream
    ↓
TCP Server
    ↓
Collector
```

The required shared components will be recreated in this notebook so that it can run independently without relying on the previous Colab runtime.

In [1]:
!apt-get update -qq
!apt-get install -y -qq build-essential g++ cmake ninja-build

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ninja-build.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.11.1-2_amd64.deb ...
Unpacking ninja-build (1.11.1-2) ...
Setting up ninja-build (1.11.1-2) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
!apt-get update -qq
!apt-get install -y -qq build-essential g++ cmake ninja-build git

!mkdir -p distributed-profiler/include
!mkdir -p distributed-profiler/src
!mkdir -p distributed-profiler/tests
!mkdir -p distributed-profiler/benchmarks
!mkdir -p distributed-profiler/apps

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
%%writefile distributed-profiler/include/profile_event.hpp

#pragma once

#include <cstdint>
#include <string>
#include <variant>

struct KernelEvent {
    std::uint64_t timestamp_ns;
    std::uint64_t duration_ns;
    std::uint32_t device_id;
    std::string kernel_name;
};

struct MemoryEvent {
    std::uint64_t timestamp_ns;
    std::uint32_t device_id;
    std::uint64_t bytes;
    std::string operation;
};

using ProfileEvent = std::variant<KernelEvent, MemoryEvent>;

Writing distributed-profiler/include/profile_event.hpp


In [4]:
%%writefile distributed-profiler/include/serializer.hpp

#pragma once

#include "profile_event.hpp"

#include <cstddef>
#include <vector>

class ISerializer {
public:
    virtual ~ISerializer() = default;

    virtual std::vector<std::byte>
    serialize(const ProfileEvent& event) const = 0;
};

Writing distributed-profiler/include/serializer.hpp


In [5]:
%%writefile distributed-profiler/include/simple_text_serializer.hpp

#pragma once

#include "serializer.hpp"

class SimpleTextSerializer : public ISerializer {
public:
    std::vector<std::byte>
    serialize(const ProfileEvent& event) const override;
};

Writing distributed-profiler/include/simple_text_serializer.hpp


In [6]:
%%writefile distributed-profiler/src/simple_text_serializer.cpp

#include "simple_text_serializer.hpp"

#include <string>
#include <type_traits>
#include <variant>

std::vector<std::byte>
SimpleTextSerializer::serialize(const ProfileEvent& event) const {
    std::string text;

    std::visit(
        [&](const auto& e) {
            using T = std::decay_t<decltype(e)>;

            if constexpr (std::is_same_v<T, KernelEvent>) {
                text =
                    "KERNEL|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.duration_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    e.kernel_name;
            }
            else if constexpr (std::is_same_v<T, MemoryEvent>) {
                text =
                    "MEMORY|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    std::to_string(e.bytes) + "|" +
                    e.operation;
            }
        },
        event
    );

    std::vector<std::byte> bytes;
    bytes.reserve(text.size());

    for (char c : text) {
        bytes.push_back(static_cast<std::byte>(c));
    }

    return bytes;
}

Writing distributed-profiler/src/simple_text_serializer.cpp


In [7]:
%%writefile distributed-profiler/include/deserializer.hpp

#pragma once

#include "profile_event.hpp"

#include <cstddef>
#include <optional>
#include <vector>

class IDeserializer {
public:
    // Add a virtual destructor.
    virtual ~IDeserializer() = default;


    // Declare a pure virtual deserialize() function.
    //
    // Input:
    //   const std::vector<std::byte>&
    //
    // Output:
    //   std::optional<ProfileEvent>
    //
    // This function should not modify the deserializer object.
    virtual std::optional<ProfileEvent>
    deserialize(
        const std::vector<std::byte>& data
    ) const = 0;
};

Writing distributed-profiler/include/deserializer.hpp


In [8]:
%%writefile distributed-profiler/include/simple_text_deserializer.hpp

#pragma once

#include "deserializer.hpp"

class SimpleTextDeserializer : public IDeserializer {
public:
    std::optional<ProfileEvent>
    deserialize(
        const std::vector<std::byte>& data
    ) const override;
};

Writing distributed-profiler/include/simple_text_deserializer.hpp


In [9]:
%%writefile distributed-profiler/src/simple_text_deserializer.cpp

#include "simple_text_deserializer.hpp"

#include <cstddef>
#include <sstream>
#include <string>
#include <vector>

std::optional<ProfileEvent>
SimpleTextDeserializer::deserialize(
    const std::vector<std::byte>& data
) const {

    // Convert std::vector<std::byte> into std::string.
    //
    // Example idea:
    //
    // std::string text;
    // for (...) {
    //     ...
    // }
    std::string text;
    text.reserve(data.size());

    for(std::byte b : data){
        text.push_back(static_cast<char>(b));
    }

    std::vector<std::string> tokens;

    std::stringstream ss(text);
    std::string token;


    // Split the text using "|" as the delimiter.
    //
    // Store fields in:
    //
    // std::vector<std::string> tokens;
    while (std::getline(ss, token, '|')) {
        tokens.push_back(token);
    }


    // If tokens is empty:
    //
    // return std::nullopt;
    if (tokens.empty()) {
        return std::nullopt;
    }

    // Check tokens[0].
    //
    // If:
    //
    // tokens[0] == "KERNEL"
    //
    // validate that the correct number of fields exists.


    // Convert kernel numeric fields:
    //
    // timestamp_ns
    // duration_ns
    // device_id
    //
    // Then construct a KernelEvent.

    // Return the KernelEvent as ProfileEvent.

    // Otherwise, if:
    //
    // tokens[0] == "MEMORY"
    //
    // validate field count.

    // Convert:
    //
    // timestamp_ns
    // device_id
    // bytes
    //
    // Then construct a MemoryEvent.


    // Return the MemoryEvent as ProfileEvent.

    // If the event type is unknown:
    //
    // return std::nullopt;
        try {
        if (tokens[0] == "KERNEL") {
            if (tokens.size() != 5) {
                return std::nullopt;
            }

            std::uint64_t timestamp_ns =
                std::stoull(tokens[1]);

            std::uint64_t duration_ns =
                std::stoull(tokens[2]);

            std::uint32_t device_id =
                static_cast<std::uint32_t>(
                    std::stoul(tokens[3])
                );

            std::string kernel_name =
                tokens[4];

            KernelEvent event{
                timestamp_ns,
                duration_ns,
                device_id,
                kernel_name
            };

            return ProfileEvent{event};
        }

        if (tokens[0] == "MEMORY") {
            if (tokens.size() != 5) {
                return std::nullopt;
            }

            std::uint64_t timestamp_ns =
                std::stoull(tokens[1]);

            std::uint32_t device_id =
                static_cast<std::uint32_t>(
                    std::stoul(tokens[2])
                );

            std::uint64_t bytes =
                std::stoull(tokens[3]);

            std::string operation =
                tokens[4];

            MemoryEvent event{
                timestamp_ns,
                device_id,
                bytes,
                operation
            };

            return ProfileEvent{event};
        }

        return std::nullopt;}


    // Consider invalid numeric conversions.
    //
    // For now, you can use try/catch around:
    //
    // std::stoull(...)
    // std::stoul(...)
    //
    // and return std::nullopt if conversion fails.
    catch (...) {
        return std::nullopt;
    }
}

Writing distributed-profiler/src/simple_text_deserializer.cpp


In [10]:
%%writefile distributed-profiler/include/frame_codec.hpp

#pragma once

#include <cstddef>
#include <optional>
#include <vector>

class FrameCodec {
public:
    // Add a function:
    //
    // std::vector<std::byte>
    // frame(const std::vector<std::byte>& payload) const;
    //
    // It should:
    // - take a payload
    // - prepend a fixed-size length field
    // - return the complete frame
    std::vector<std::byte>
    frame(const std::vector<std::byte>& payload) const;


    // Add a function:
    //
    // std::optional<std::vector<std::byte>>
    // unframe(const std::vector<std::byte>& frame) const;
    //
    // It should:
    // - read the length field
    // - validate the frame size
    // - extract and return the payload
    // - return std::nullopt if the frame is malformed
    std::optional<std::vector<std::byte>>
    unframe(const std::vector<std::byte>& frame) const;
};

Writing distributed-profiler/include/frame_codec.hpp


In [11]:
%%writefile distributed-profiler/src/frame_codec.cpp

#include "frame_codec.hpp"

#include <cstdint>
#include <cstring>

/*
frame():

payload
   ↓
get payload.size()
   ↓
store size as uint32_t
   ↓
allocate:
4 bytes + payload bytes
   ↓
copy size into first 4 bytes
   ↓
copy payload after that


+--------------------+----------------------+
| uint32_t length    | payload              |
| 4 bytes            | N bytes              |
+--------------------+----------------------+

unframe()
frame
   ↓
check at least 4 bytes
   ↓
read uint32_t length
   ↓
calculate expected total size
   ↓
validate size
   ↓
extract payload

 */

std::vector<std::byte>
FrameCodec::frame(
    const std::vector<std::byte>& payload
) const {
    // Store payload.size() in a std::uint32_t.
    std::uint32_t payload_size =
        static_cast<std::uint32_t>(payload.size());

    // Create an output vector.
    //
    // Total size should be:
    //
    // sizeof(std::uint32_t) + payload.size()
    std::vector<std::byte> framed_data(
        sizeof(std::uint32_t) + payload.size()
    );


    // Append the 4-byte length prefix.
    std::memcpy(
        framed_data.data(),
        &payload_size,
        sizeof(std::uint32_t)
    );


    // Append all payload bytes.
    std::memcpy(
        framed_data.data() + sizeof(std::uint32_t),
        payload.data(),
        payload.size()
    );

    // Return the complete frame.
    return framed_data;
}


std::optional<std::vector<std::byte>>
FrameCodec::unframe(
    const std::vector<std::byte>& frame
) const {
    // First check that frame contains at least
    // sizeof(std::uint32_t) bytes.
    //
    // Otherwise return std::nullopt.
    if (frame.size() < sizeof(std::uint32_t)) {
        return std::nullopt;
    }

    // Read the first 4 bytes into a std::uint32_t
    // representing payload_size.
    std::uint32_t payload_size = 0;


    // Compute the expected total frame size:
    //
    // sizeof(std::uint32_t) + payload_size
    std::memcpy(
        &payload_size,
        frame.data(),
        sizeof(std::uint32_t)
    );


    // Validate that frame.size() matches the
    // expected total size.
    //
    // If not:
    // return std::nullopt;
    std::size_t expected_size =
        sizeof(std::uint32_t) +
        static_cast<std::size_t>(payload_size);

    if (frame.size() != expected_size) {
        return std::nullopt;
    }

    // Extract payload bytes into a new vector.
    std::vector<std::byte> payload(
        frame.begin() + sizeof(std::uint32_t),
        frame.end()
    );

    // Return the payload.
        return payload;
}

Writing distributed-profiler/src/frame_codec.cpp


In [12]:
%%writefile distributed-profiler/include/batch_builder.hpp

#pragma once

#include <cstddef>
#include <vector>

class BatchBuilder {
public:

    // Add a function to append one framed event.
    //
    // Input:
    //   const std::vector<std::byte>& frame
    //
    // Behavior:
    //   append all bytes from frame to the internal batch buffer.
    void append(
        const std::vector<std::byte>& frame
    );

    // Add a function that returns the current batch buffer.
    const std::vector<std::byte>&
    data() const;

    void clear();


    // Add a size() function that returns
    // the current batch size in bytes.
    std::size_t
    size() const;

private:
    // Store the batch bytes here.
    std::vector<std::byte> buffer_;
};

Writing distributed-profiler/include/batch_builder.hpp


In [13]:
%%writefile distributed-profiler/src/batch_builder.cpp

#include "batch_builder.hpp"

void BatchBuilder::append(
    const std::vector<std::byte>& frame
) {
    // Append all bytes from frame into buffer_.
    buffer_.insert(
        buffer_.end(),
        frame.begin(),
        frame.end()
    );
}


const std::vector<std::byte>&
BatchBuilder::data() const {
    return buffer_;
}


void BatchBuilder::clear() {
    buffer_.clear();
}


std::size_t
BatchBuilder::size() const {
    return buffer_.size();
}

Writing distributed-profiler/src/batch_builder.cpp


In [14]:
%%writefile distributed-profiler/include/batch_parser.hpp

#pragma once

#include <cstddef>
#include <optional>
#include <vector>

class BatchParser {
public:
    // Add a parse() function.
    //
    // Input:
    //   const std::vector<std::byte>& batch
    //
    // Output:
    //   std::optional<
    //       std::vector<std::vector<std::byte>>
    //   >
    //
    // Each inner vector should contain one COMPLETE FRAME.
    //
    // If the batch is malformed:
    //   return std::nullopt;
    std::optional<
        std::vector<std::vector<std::byte>>
    >
    parse(
        const std::vector<std::byte>& batch
    ) const;
};

Writing distributed-profiler/include/batch_parser.hpp


In [15]:
%%writefile distributed-profiler/src/batch_parser.cpp

#include "batch_parser.hpp"

#include <cstdint>
#include <cstring>
/*offset = 0
   ↓
read payload_size
   ↓
frame_size = 4 + payload_size
   ↓
check enough bytes remain
   ↓
extract complete frame
   ↓
frames.push_back(...)
   ↓
offset += frame_size
   ↓
repeat*/

std::optional<
    std::vector<std::vector<std::byte>>
>
BatchParser::parse(
    const std::vector<std::byte>& batch
) const {

    std::vector<std::vector<std::byte>> frames;

    std::size_t offset = 0;

    // Loop while offset < batch.size().
       while (offset < batch.size()) {

        // At least 4 bytes must remain for the length prefix.
        // Check that at least 4 bytes remain
        // for the uint32_t length prefix.
        //
        // If not:
        //     return std::nullopt;
        if (
            batch.size() - offset <
            sizeof(std::uint32_t)
        ) {
            return std::nullopt;
        }

        // Read payload_size from:
        //
        // batch.data() + offset
        //
        // using std::memcpy.
        std::uint32_t payload_size = 0;
        std::memcpy(
            &payload_size,
            batch.data() + offset,
            sizeof(std::uint32_t)
        );

        // Calculate total frame size:
        //
        // sizeof(std::uint32_t) + payload_size
        std::size_t frame_size =
            sizeof(std::uint32_t) +
            static_cast<std::size_t>(payload_size);


        // Check that enough bytes remain in the batch.
        //
        // If:
        //
        // offset + frame_size > batch.size()
        //
        // return std::nullopt;
        if (frame_size > batch.size() - offset) {
            return std::nullopt;
        }


        // Extract the COMPLETE frame:
        //
        // [length][payload]
        //
        // into a new std::vector<std::byte>.
        auto frame_begin =
            batch.begin() + offset;

        auto frame_end =
            frame_begin + frame_size;

        std::vector<std::byte> frame(
            frame_begin,
            frame_end
        );


        // Push that frame into frames.
        frames.push_back(
            std::move(frame)
        );


        // Advance offset by frame_size.
        offset += frame_size;}


    // After the loop finishes,
    // return frames.
    return frames;
}

Writing distributed-profiler/src/batch_parser.cpp


## Step 2 — Basic TCP Client / Server

Before sending real profiling events, the TCP connection itself should be validated independently.

The goal of this step is to establish a simple client-server connection over localhost and send a small text message.

The architecture is:

```text
Profiler Agent / Client
        ↓
      socket()
        ↓
     connect()
        ↓
      send()
        ↓
======================
      TCP Stream
======================
        ↓
      accept()
        ↓
      recv()
        ↓
Collector / Server
```

---

## Server Responsibilities

The collector acts as the TCP server.

Its basic lifecycle is:

```text
socket()
    ↓
bind()
    ↓
listen()
    ↓
accept()
    ↓
recv()
    ↓
close()
```

### `socket()`

Creates a socket endpoint.

Conceptually:

```text
create a communication endpoint
```

For IPv4 TCP communication, the socket will use:

```text
AF_INET
SOCK_STREAM
```

---

### `bind()`

Associates the server socket with a local IP address and port.

For this notebook, the server will listen on localhost:

```text
127.0.0.1
```

using a port such as:

```text
19000
```

Conceptually:

```text
socket
    ↓
bind to 127.0.0.1:19000
```

---

### `listen()`

Marks the socket as a listening socket.

The server is now ready to receive incoming connection requests.

Conceptually:

```text
server socket
    ↓
listen()
    ↓
waiting for clients
```

---

### `accept()`

Waits for a client to connect.

When a client connects, `accept()` creates a new connected socket that is used for communication with that client.

This means the server normally has:

```text
listening socket
```

and:

```text
connected client socket
```

as two separate file descriptors.

---

## Client Responsibilities

The profiler agent acts as the TCP client.

Its lifecycle is:

```text
socket()
    ↓
connect()
    ↓
send()
    ↓
close()
```

The client creates a socket and connects to:

```text
127.0.0.1:19000
```

Once connected, it sends a simple test message:

```text
Hello from profiler agent
```

---

## Basic Communication Flow

The execution order is:

```text
Server starts
    ↓
socket()
    ↓
bind()
    ↓
listen()
    ↓
accept()
    ↓
server waits
```

Then the client starts:

```text
Client starts
    ↓
socket()
    ↓
connect()
```

The connection is established:

```text
Client                           Server

connect()  ------------------>  accept()
```

Then the client sends data:

```text
send("Hello from profiler agent")
        ↓
      TCP
        ↓
recv(...)
```

The server prints the received message.

---

## First Validation Target

For this step, no serialization or framing is required.

The data is simply:

```text
Hello from profiler agent
```

The expected output on the server side is:

```text
Server received: Hello from profiler agent
```

This proves that:

- the server socket was created correctly
- `bind()` succeeded
- `listen()` succeeded
- `accept()` succeeded
- the client connected successfully
- data moved through TCP
- the server received the transmitted bytes

---

## Why Start With Plain Text?

The final project will eventually send:

```text
ProfileEvent
    ↓
serialize
    ↓
frame
    ↓
batch
    ↓
TCP
```

However, debugging networking and serialization at the same time would make failures harder to isolate.

Therefore, this step validates only:

```text
TCP connectivity
```

before adding:

```text
serialization
framing
batching
```

---

## Localhost Testing

Both client and server will run inside the same Colab/Linux environment.

The connection will use:

```text
127.0.0.1
```

which means:

```text
localhost
```

No external network connection is required.

The data still passes through the operating system TCP stack, so the experiment exercises real socket APIs even though both processes run on the same machine.

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- What is a TCP client?
- What is a TCP server?
- What does `socket()` do?
- What does `bind()` do?
- What does `listen()` do?
- What does `accept()` do?
- What does `connect()` do?
- What do `send()` and `recv()` do?
- Why does the server have both a listening socket and a connected socket?
- Why validate plain TCP before adding profiling data?

In [16]:
%%writefile distributed-profiler/apps/basic_tcp_server.cpp

#include <arpa/inet.h>
#include <cstring>
#include <iostream>
#include <sys/socket.h>
#include <unistd.h>

int main() {
    const int port = 19000;

    // 1. Create a TCP socket.
    int server_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (server_fd < 0) {
        std::cerr << "socket() failed\n";
        return 1;
    }

    // 2. Create and initialize the server address.
    sockaddr_in server_addr{};

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(port);

    if (
        inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_addr.sin_addr
        ) <= 0
    ) {
        std::cerr << "inet_pton() failed\n";
        close(server_fd);
        return 1;
    }

    // 3. Bind the socket.
    if (
        bind(
            server_fd,
            reinterpret_cast<sockaddr*>(&server_addr),
            sizeof(server_addr)
        ) < 0
    ) {
        std::cerr << "bind() failed\n";
        close(server_fd);
        return 1;
    }

    // 4. Put the socket into listening mode.
    if (listen(server_fd, 1) < 0) {
        std::cerr << "listen() failed\n";
        close(server_fd);
        return 1;
    }

    std::cout
        << "Server listening on 127.0.0.1:"
        << port
        << '\n';

    // 5. Wait for one client connection.
    int client_fd = accept(
        server_fd,
        nullptr,
        nullptr
    );

    if (client_fd < 0) {
        std::cerr << "accept() failed\n";
        close(server_fd);
        return 1;
    }

    std::cout << "Client connected\n";

    // 6. Create receive buffer.
    char buffer[1024] = {0};

    // 7. Receive data.
    ssize_t bytes_received = recv(
        client_fd,
        buffer,
        sizeof(buffer) - 1,
        0
    );

    if (bytes_received < 0) {
        std::cerr << "recv() failed\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }

    // 8. Print received data.
    if (bytes_received > 0) {
        buffer[bytes_received] = '\0';

        std::cout
            << "Server received: "
            << buffer
            << '\n';
    }
    else {
        std::cout
            << "Client closed the connection\n";
    }

    // 9. Close connected socket.
    close(client_fd);

    // 10. Close listening socket.
    close(server_fd);

    return 0;
}

Writing distributed-profiler/apps/basic_tcp_server.cpp


In [17]:
%%writefile distributed-profiler/apps/basic_tcp_client.cpp

#include <arpa/inet.h>
#include <cstring>
#include <iostream>
#include <string>
#include <sys/socket.h>
#include <unistd.h>

int main() {
    const int port = 19000;

    // 1. Create a TCP socket.
    int socket_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (socket_fd < 0) {
        std::cerr << "socket() failed\n";
        return 1;
    }

    // 2. Create and initialize the server address.
    sockaddr_in server_addr{};

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(port);

    if (
        inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_addr.sin_addr
        ) <= 0
    ) {
        std::cerr << "inet_pton() failed\n";
        close(socket_fd);
        return 1;
    }

    // 3. Connect to the server.
    if (
        connect(
            socket_fd,
            reinterpret_cast<sockaddr*>(&server_addr),
            sizeof(server_addr)
        ) < 0
    ) {
        std::cerr << "connect() failed\n";
        close(socket_fd);
        return 1;
    }

    // 4. Create test message.
    std::string message =
        "Hello from profiler agent";

    // 5. Send the message.
    ssize_t bytes_sent = send(
        socket_fd,
        message.data(),
        message.size(),
        0
    );

    if (bytes_sent < 0) {
        std::cerr << "send() failed\n";
        close(socket_fd);
        return 1;
    }

    // 6. Print what was sent.
    std::cout
        << "Client sent: "
        << message
        << '\n';

    std::cout
        << "Bytes sent: "
        << bytes_sent
        << '\n';

    // 7. Close the socket.
    close(socket_fd);

    return 0;
}

Writing distributed-profiler/apps/basic_tcp_client.cpp


In [18]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    distributed-profiler/apps/basic_tcp_server.cpp \
    -o basic_tcp_server

!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    distributed-profiler/apps/basic_tcp_client.cpp \
    -o basic_tcp_client

In [19]:
!./basic_tcp_server > server_output.txt 2>&1 & \
SERVER_PID=$!; \
sleep 1; \
./basic_tcp_client; \
wait $SERVER_PID; \
cat server_output.txt

Client sent: Hello from profiler agent
Bytes sent: 25
Server listening on 127.0.0.1:19000
Client connected
Server received: Hello from profiler agent


## Step 3 — Send One Framed Profiling Event over TCP

The previous step validated basic TCP connectivity using a plain text message:

```text
Client
    ↓
"Hello from profiler agent"
    ↓
TCP
    ↓
Server
```

The goal of this step is to replace the plain text message with a real profiling event.

This connects the profiling data pipeline from the previous notebook with the TCP transport layer.

---

## Target Data Flow

The client acts as the profiler agent.

It creates a profiling event:

```text
KernelEvent
    ↓
ProfileEvent
```

Then it serializes and frames the event:

```text
ProfileEvent
    ↓
SimpleTextSerializer
    ↓
serialized payload
    ↓
FrameCodec
    ↓
[length][payload]
```

The framed bytes are then sent over TCP:

```text
[length][payload]
    ↓
send()
    ↓
========================
       TCP Stream
========================
    ↓
recv()
```

The server acts as the collector.

It receives the framed bytes and reconstructs the event:

```text
recv()
    ↓
[length][payload]
    ↓
FrameCodec::unframe()
    ↓
serialized payload
    ↓
SimpleTextDeserializer
    ↓
ProfileEvent
```

The complete path is:

```text
Profiler Agent / Client

KernelEvent
    ↓
ProfileEvent
    ↓
serialize()
    ↓
payload bytes
    ↓
frame()
    ↓
[length][payload]
    ↓
send()
    ↓
========================
       TCP Stream
========================
    ↓
recv()
    ↓
[length][payload]
    ↓
unframe()
    ↓
payload bytes
    ↓
deserialize()
    ↓
ProfileEvent

Collector / Server
```

---

## Client-Side Responsibilities

The client will create one profiling event:

```cpp
KernelEvent{
    1000,
    200,
    0,
    "gemm_kernel"
}
```

It will then store the event inside:

```cpp
ProfileEvent
```

and serialize it using:

```cpp
SimpleTextSerializer
```

The serialized representation will be:

```text
KERNEL|1000|200|0|gemm_kernel
```

The payload is then passed to:

```cpp
FrameCodec::frame()
```

which produces:

```text
[length][payload]
```

Finally, the client sends the complete framed buffer through the TCP socket.

---

## Server-Side Responsibilities

The server receives the TCP bytes into a buffer.

For this step, the received bytes should represent one complete frame:

```text
[length][payload]
```

The server passes this buffer to:

```cpp
FrameCodec::unframe()
```

to recover the serialized payload.

Then:

```cpp
SimpleTextDeserializer
```

reconstructs the original:

```cpp
ProfileEvent
```

The server can use:

```cpp
std::visit()
```

to inspect the concrete event type and print its fields.

---

## Expected Result

The client sends:

```text
KernelEvent
timestamp = 1000
duration  = 200
device    = 0
kernel    = gemm_kernel
```

The server should reconstruct exactly:

```text
KernelEvent
timestamp = 1000
duration  = 200
device    = 0
kernel    = gemm_kernel
```

This proves that the full path works:

```text
ProfileEvent
    ↓
serialization
    ↓
framing
    ↓
TCP transport
    ↓
unframing
    ↓
deserialization
    ↓
ProfileEvent
```

---

## Simplifying Assumption

For this step, we will temporarily assume that:

```text
one send()
```

transmits the full frame and that:

```text
one recv()
```

receives the full frame.

This assumption is useful for isolating the integration between:

```text
serialization
framing
TCP
deserialization
```

but it is not guaranteed by TCP.

A later step will remove this assumption by implementing:

```text
send_all()
```

and:

```text
recv_exact()
```

to handle partial network I/O correctly.

---

## Files Used in This Step

The client will use:

```text
profile_event.hpp
serializer.hpp
simple_text_serializer.hpp
frame_codec.hpp
```

The server will use:

```text
profile_event.hpp
deserializer.hpp
simple_text_deserializer.hpp
frame_codec.hpp
```

The new application files will be:

```text
apps/profile_event_client.cpp
apps/profile_event_server.cpp
```

---

## Validation Criteria

This step is successful if:

```text
1. Server starts and listens.
2. Client connects successfully.
3. Client creates one ProfileEvent.
4. Client serializes the event.
5. Client frames the payload.
6. Client sends the frame.
7. Server receives the frame.
8. Server unframes the payload.
9. Server deserializes the event.
10. Reconstructed fields match the original event.
```

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- How is a `ProfileEvent` prepared for network transmission?
- Why should serialization happen before framing?
- What exactly is sent through the TCP socket?
- How does the server recover the framed payload?
- How does the collector reconstruct the original `ProfileEvent`?
- Why is this still not production-safe with only one `send()` and one `recv()`?
- What problem will `send_all()` and `recv_exact()` solve in the next step?

In [25]:
%%writefile distributed-profiler/apps/profile_event_client.cpp

#include "frame_codec.hpp"
#include "profile_event.hpp"
#include "simple_text_serializer.hpp"

#include <arpa/inet.h>
#include <iostream>
#include <sys/socket.h>
#include <unistd.h>

int main() {
    const int port = 19001;

    SimpleTextSerializer serializer;
    FrameCodec codec;

    // 1. Create TCP socket.
    int socket_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (socket_fd < 0) {
        std::cerr << "socket() failed\n";
        return 1;
    }

    // 2. Create server address.
    sockaddr_in server_addr{};

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(port);

    if (
        inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_addr.sin_addr
        ) <= 0
    ) {
        std::cerr << "inet_pton() failed\n";
        close(socket_fd);
        return 1;
    }

    // 3. Connect to server.
    if (
        connect(
            socket_fd,
            reinterpret_cast<sockaddr*>(&server_addr),
            sizeof(server_addr)
        ) < 0
    ) {
        std::cerr << "connect() failed\n";
        close(socket_fd);
        return 1;
    }

    // 4. Create one KernelEvent.
    KernelEvent kernel{
        1000,
        200,
        0,
        "gemm_kernel"
    };

    // 5. Store it inside ProfileEvent.
    ProfileEvent event = kernel;

    // 6. Serialize the event.
    auto payload =
        serializer.serialize(event);

    // 7. Add length-prefix framing.
    auto frame =
        codec.frame(payload);

    // 8. Send the complete frame.
    //
    // For Step 3 only, we assume send()
    // transmits the complete buffer.
    ssize_t bytes_sent = send(
        socket_fd,
        frame.data(),
        frame.size(),
        0
    );

    // 9. Check for send failure.
    if (bytes_sent < 0) {
        std::cerr << "send() failed\n";
        close(socket_fd);
        return 1;
    }

    // 10. Print statistics.
    std::cout
        << "Payload size: "
        << payload.size()
        << " bytes\n";

    std::cout
        << "Frame size: "
        << frame.size()
        << " bytes\n";

    std::cout
        << "Bytes sent: "
        << bytes_sent
        << '\n';

    // Optional validation for this simplified step.
    if (
        static_cast<std::size_t>(bytes_sent)
        != frame.size()
    ) {
        std::cerr
            << "Warning: partial send occurred\n";
    }

    // 11. Close socket.
    close(socket_fd);

    return 0;
}

Overwriting distributed-profiler/apps/profile_event_client.cpp


In [26]:
%%writefile distributed-profiler/apps/profile_event_server.cpp

#include "frame_codec.hpp"
#include "profile_event.hpp"
#include "simple_text_deserializer.hpp"

#include <arpa/inet.h>
#include <cstddef>
#include <iostream>
#include <sys/socket.h>
#include <type_traits>
#include <unistd.h>
#include <variant>
#include <vector>

int main() {
    const int port = 19001;

    SimpleTextDeserializer deserializer;
    FrameCodec codec;

    // 1. Create server TCP socket.
    int server_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (server_fd < 0) {
        std::cerr << "socket() failed\n";
        return 1;
    }

    // 2. Create server address.
    sockaddr_in server_addr{};

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(port);

    if (
        inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_addr.sin_addr
        ) <= 0
    ) {
        std::cerr << "inet_pton() failed\n";
        close(server_fd);
        return 1;
    }

    // 3. Bind socket.
    if (
        bind(
            server_fd,
            reinterpret_cast<sockaddr*>(&server_addr),
            sizeof(server_addr)
        ) < 0
    ) {
        std::cerr << "bind() failed\n";
        close(server_fd);
        return 1;
    }

    // 4. Listen for connections.
    if (listen(server_fd, 1) < 0) {
        std::cerr << "listen() failed\n";
        close(server_fd);
        return 1;
    }

    std::cout
        << "Server listening on 127.0.0.1:"
        << port
        << '\n';

    // 5. Accept one client.
    int client_fd = accept(
        server_fd,
        nullptr,
        nullptr
    );

    if (client_fd < 0) {
        std::cerr << "accept() failed\n";
        close(server_fd);
        return 1;
    }

    std::cout << "Client connected\n";

    // 6. Create receive buffer.
    std::vector<std::byte> buffer(1024);

    // 7. Receive bytes.
    ssize_t bytes_received = recv(
        client_fd,
        buffer.data(),
        buffer.size(),
        0
    );

    // 8. Check receive result.
    if (bytes_received < 0) {
        std::cerr << "recv() failed\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }

    if (bytes_received == 0) {
        std::cout << "Client disconnected\n";

        close(client_fd);
        close(server_fd);

        return 0;
    }

    // 9. Resize to actual number of bytes received.
    buffer.resize(
        static_cast<std::size_t>(bytes_received)
    );

    std::cout
        << "Server received "
        << buffer.size()
        << " bytes\n";

    // 10. Remove length-prefix framing.
    auto payload =
        codec.unframe(buffer);

    if (!payload.has_value()) {
        std::cerr << "Unframing failed\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }

    // 11. Deserialize payload.
    auto event =
        deserializer.deserialize(*payload);

    if (!event.has_value()) {
        std::cerr << "Deserialization failed\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }

    // 12. Print reconstructed event.
    std::visit(
        [](const auto& e) {
            using T =
                std::decay_t<decltype(e)>;

            if constexpr (
                std::is_same_v<T, KernelEvent>
            ) {
                std::cout
                    << "Received KernelEvent\n"
                    << "kernel=" << e.kernel_name << '\n'
                    << "timestamp=" << e.timestamp_ns << '\n'
                    << "duration=" << e.duration_ns << '\n'
                    << "device=" << e.device_id << '\n';
            }
            else if constexpr (
                std::is_same_v<T, MemoryEvent>
            ) {
                std::cout
                    << "Received MemoryEvent\n"
                    << "operation=" << e.operation << '\n'
                    << "timestamp=" << e.timestamp_ns << '\n'
                    << "bytes=" << e.bytes << '\n'
                    << "device=" << e.device_id << '\n';
            }
        },
        *event
    );

    // 13. Close client socket.
    close(client_fd);

    // 14. Close listening socket.
    close(server_fd);

    return 0;
}

Overwriting distributed-profiler/apps/profile_event_server.cpp


In [27]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/frame_codec.cpp \
    distributed-profiler/src/simple_text_deserializer.cpp \
    distributed-profiler/apps/profile_event_server.cpp \
    -o profile_event_server

In [28]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/frame_codec.cpp \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/apps/profile_event_client.cpp \
    -o profile_event_client

In [29]:
!./profile_event_server > profile_server_output.txt 2>&1 & \
SERVER_PID=$!; \
sleep 1; \
./profile_event_client; \
wait $SERVER_PID; \
cat profile_server_output.txt

Payload size: 29 bytes
Frame size: 33 bytes
Bytes sent: 33
Server listening on 127.0.0.1:19001
Client connected
Server received 33 bytes
Received KernelEvent
kernel=gemm_kernel
timestamp=1000
duration=200
device=0


## Step 4 — Reliable `send_all()` and `recv_exact()`

The previous step successfully sent one framed profiling event over TCP:

```text
ProfileEvent
    ↓
serialize()
    ↓
frame()
    ↓
send()
    ↓
TCP
    ↓
recv()
    ↓
unframe()
    ↓
deserialize()
```

However, the previous implementation made one simplifying assumption:

```text
one send() sends the entire buffer
```

and:

```text
one recv() receives the entire frame
```

TCP does not guarantee either behavior.

---

## Why One `send()` May Not Send Everything

Suppose the client wants to send:

```text
100 bytes
```

using:

```cpp
send(fd, buffer, 100, 0);
```

The function may return:

```text
40
```

This means only 40 bytes were accepted by the operating system.

The remaining:

```text
60 bytes
```

still need to be sent.

Conceptually:

```text
100 bytes to send
      ↓
send()
      ↓
40 bytes sent
      ↓
60 bytes remain
      ↓
send() again
```

Therefore, reliable transport code must continue calling `send()` until all requested bytes have been transmitted.

---

## `send_all()`

A helper function can hide this repeated-send logic.

Conceptually:

```cpp
bool send_all(
    int fd,
    const std::byte* data,
    std::size_t size
);
```

The function keeps track of:

```text
total bytes sent
```

and repeatedly sends:

```text
remaining bytes
```

until:

```text
total_sent == size
```

Conceptually:

```text
total_sent = 0

while total_sent < size
    ↓
send(data + total_sent,
     size - total_sent)
    ↓
add bytes_sent to total_sent
```

If `send()` fails, the helper returns:

```cpp
false
```

Otherwise:

```cpp
true
```

means the complete buffer was transmitted.

---

## Why One `recv()` May Not Receive Everything

The same issue exists on the receiving side.

Suppose the server wants:

```text
100 bytes
```

and calls:

```cpp
recv(fd, buffer, 100, 0);
```

The operating system may return:

```text
32 bytes
```

even though more data is still coming.

The next call might return:

```text
50 bytes
```

and another:

```text
18 bytes
```

Together:

```text
32 + 50 + 18 = 100 bytes
```

Therefore, receiving a complete message may require multiple calls to `recv()`.

---

## `recv_exact()`

A helper function can repeatedly call `recv()` until an exact number of bytes has been received.

Conceptually:

```cpp
bool recv_exact(
    int fd,
    std::byte* buffer,
    std::size_t size
);
```

The logic is:

```text
total_received = 0

while total_received < size
    ↓
recv(buffer + total_received,
     size - total_received)
    ↓
add bytes_received
```

The function returns:

```text
true
```

only when exactly the requested number of bytes has been received.

---

## Connection Closure

For `recv()`, a return value of:

```text
0
```

has a special meaning.

It means:

```text
the peer closed the TCP connection
```

Therefore:

```text
recv() < 0
    → error

recv() == 0
    → peer disconnected

recv() > 0
    → bytes received
```

A reliable `recv_exact()` helper must handle all three cases.

---

## Using Framing with `recv_exact()`

The profiling protocol already uses:

```text
[length][payload]
```

This fits naturally with exact-length receiving.

The server can reconstruct one message in two stages.

### Stage 1 — Receive the Header

First receive exactly:

```text
4 bytes
```

for the `std::uint32_t` payload length.

Conceptually:

```text
recv_exact(4 bytes)
        ↓
decode payload_size
```

For example:

```text
payload_size = 29
```

---

### Stage 2 — Receive the Payload

Now the server knows exactly how many payload bytes must follow:

```text
recv_exact(29 bytes)
```

The result is:

```text
KERNEL|1000|200|0|gemm_kernel
```

So the complete receive process becomes:

```text
TCP stream
    ↓
recv_exact(4-byte header)
    ↓
decode payload length N
    ↓
allocate N-byte buffer
    ↓
recv_exact(N bytes)
    ↓
complete payload
```

---

## Why This Is Better Than a Large Receive Buffer

The previous version used something like:

```cpp
std::vector<std::byte> buffer(1024);

recv(
    client_fd,
    buffer.data(),
    buffer.size(),
    0
);
```

This only means:

```text
receive up to 1024 bytes
```

It does not mean:

```text
receive one complete application message
```

By combining:

```text
length-prefix framing
```

with:

```text
recv_exact()
```

the application can reconstruct messages correctly regardless of how TCP divides the byte stream internally.

---

## Updated Client Flow

The client-side path becomes:

```text
ProfileEvent
    ↓
serialize()
    ↓
frame()
    ↓
send_all(frame.data(), frame.size())
```

Instead of:

```text
one send()
```

the helper guarantees that the whole frame is transmitted or reports failure.

---

## Updated Server Flow

The server-side path becomes:

```text
recv_exact(4-byte header)
        ↓
read payload_size
        ↓
recv_exact(payload_size bytes)
        ↓
deserialize payload
```

This removes the assumption that one `recv()` corresponds to one complete frame.

---

## Important TCP Principle

The key idea is:

```text
TCP preserves byte order,
but not application message boundaries.
```

So the application must explicitly manage:

```text
message size
partial writes
partial reads
connection closure
```

---

## Validation Target

The same profiling event should still be transmitted:

```text
KernelEvent
timestamp = 1000
duration  = 200
device    = 0
kernel    = gemm_kernel
```

but now the transport path should be:

```text
Client
    ↓
send_all()
    ↓
TCP byte stream
    ↓
recv_exact(header)
    ↓
recv_exact(payload)
    ↓
deserialize
    ↓
Collector
```

The reconstructed event should remain identical to the original event.

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- Why can `send()` return fewer bytes than requested?
- Why can `recv()` return fewer bytes than requested?
- What is a partial send?
- What is a partial receive?
- What does `recv() == 0` mean?
- How does `send_all()` guarantee complete transmission?
- How does `recv_exact()` guarantee complete reception?
- Why is length-prefix framing useful with TCP?
- Why should the receiver read the header before the payload?
- Why does TCP preserve byte order but not message boundaries?

In [30]:
%%writefile distributed-profiler/include/tcp_utils.hpp

#pragma once

#include <cstddef>

bool send_all(
    int fd,
    const std::byte* data,
    std::size_t size
);

bool recv_exact(
    int fd,
    std::byte* buffer,
    std::size_t size
);

Writing distributed-profiler/include/tcp_utils.hpp


In [31]:
%%writefile distributed-profiler/src/tcp_utils.cpp

#include "tcp_utils.hpp"

#include <sys/socket.h>


// ============================================================
// send_all()
// Goal:
// Keep calling send() until the entire buffer has been sent.
// ============================================================

bool send_all(
    int fd,
    const std::byte* data,
    std::size_t size
) {
    // Track how many bytes have already been sent.
    std::size_t total_sent = 0;


    // Keep sending until:
    //
    // total_sent == size
    //
    // because one send() call may send only
    // part of the requested buffer.
    while (total_sent < size) {


        // Send only the remaining unsent portion.
        //
        // data + total_sent:
        //     start from the first byte that
        //     has not been sent yet.
        //
        // size - total_sent:
        //     number of bytes still remaining.
        ssize_t bytes_sent = send(
            fd,
            data + total_sent,
            size - total_sent,
            0
        );

        // If send() fails, stop.
        //
        // send() normally returns:
        //
        // > 0 : number of bytes sent
        // < 0 : error
        //
        // We also treat 0 as failure here,
        // because no forward progress was made.
        if (bytes_sent <= 0) {
            return false;
        }


        // Update the progress counter.
        total_sent +=
            static_cast<std::size_t>(bytes_sent);
    }

    // If we reach this point,
    // all requested bytes have been sent.
    return true;
}



// ============================================================
// recv_exact()
// Goal:
// Keep calling recv() until exactly "size" bytes
// have been received.
// ============================================================

bool recv_exact(
    int fd,
    std::byte* buffer,
    std::size_t size
) {
    // Track how many bytes have already been received.
    std::size_t total_received = 0;


    // Keep receiving until:
    //
    // total_received == size
    //
    // because one recv() call may return only
    // part of the requested data.
    while (total_received < size) {

        // Receive into the unused part of the buffer.
        //
        // buffer + total_received:
        //     write new bytes after the bytes
        //     already received.
        //
        // size - total_received:
        //     number of bytes still needed.
        ssize_t bytes_received = recv(
            fd,
            buffer + total_received,
            size - total_received,
            0
        );


        // recv() < 0 means an error occurred.
        if (bytes_received < 0) {
            return false;
        }


        // recv() == 0 means the peer has closed
        // the TCP connection.
        //
        // If we still need more bytes,
        // receiving the complete message is impossible.
        if (bytes_received == 0) {
            return false;
        }


        // Update the number of bytes collected.
        total_received +=
            static_cast<std::size_t>(bytes_received);
    }


    // Exactly "size" bytes have now been received.
    return true;
}

Writing distributed-profiler/src/tcp_utils.cpp


In [32]:
%%writefile distributed-profiler/apps/reliable_profile_client.cpp

#include "frame_codec.hpp"
#include "profile_event.hpp"
#include "simple_text_serializer.hpp"
#include "tcp_utils.hpp"

#include <arpa/inet.h>
#include <iostream>
#include <sys/socket.h>
#include <unistd.h>

int main() {
    const int port = 19002;

    SimpleTextSerializer serializer;
    FrameCodec codec;

    // Create TCP socket.
    int socket_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (socket_fd < 0) {
        std::cerr << "socket() failed\n";
        return 1;
    }


    // Configure server address.
    sockaddr_in server_addr{};

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(port);

    if (
        inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_addr.sin_addr
        ) <= 0
    ) {
        std::cerr << "inet_pton() failed\n";
        close(socket_fd);
        return 1;
    }


    // Connect to server.
    if (
        connect(
            socket_fd,
            reinterpret_cast<sockaddr*>(&server_addr),
            sizeof(server_addr)
        ) < 0
    ) {
        std::cerr << "connect() failed\n";
        close(socket_fd);
        return 1;
    }


    // Create one KernelEvent.
    KernelEvent kernel{
        1000,
        200,
        0,
        "gemm_kernel"
    };


    // Store the concrete event inside ProfileEvent.
    ProfileEvent event = kernel;


    // Serialize ProfileEvent into raw payload bytes.
    auto payload =
        serializer.serialize(event);


    // Add the length-prefix framing.
    //
    // frame layout:
    //
    // [4-byte length][payload]
    auto frame =
        codec.frame(payload);


    // Reliably send the complete frame.
    //
    // send_all() keeps calling send()
    // until every byte has been transmitted.
    bool success = send_all(
        socket_fd,
        frame.data(),
        frame.size()
    );


    // Check whether transmission succeeded.
    if (!success) {
        std::cerr << "send_all() failed\n";
        close(socket_fd);
        return 1;
    }


    // Print useful information.
    std::cout
        << "Payload size: "
        << payload.size()
        << " bytes\n";

    std::cout
        << "Frame size: "
        << frame.size()
        << " bytes\n";

    std::cout
        << "Frame sent successfully\n";


    // Close TCP socket.
    close(socket_fd);

    return 0;
}

Writing distributed-profiler/apps/reliable_profile_client.cpp


In [33]:
%%writefile distributed-profiler/apps/reliable_profile_server.cpp

#include "profile_event.hpp"
#include "simple_text_deserializer.hpp"
#include "tcp_utils.hpp"

#include <arpa/inet.h>
#include <cstddef>
#include <cstdint>
#include <cstring>
#include <iostream>
#include <sys/socket.h>
#include <type_traits>
#include <unistd.h>
#include <variant>
#include <vector>

int main() {
    const int port = 19002;

    SimpleTextDeserializer deserializer;


    // Create server TCP socket.
    int server_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (server_fd < 0) {
        std::cerr << "socket() failed\n";
        return 1;
    }


    // Configure server address.
    sockaddr_in server_addr{};

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(port);

    if (
        inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_addr.sin_addr
        ) <= 0
    ) {
        std::cerr << "inet_pton() failed\n";
        close(server_fd);
        return 1;
    }

    // Bind server socket.
    if (
        bind(
            server_fd,
            reinterpret_cast<sockaddr*>(&server_addr),
            sizeof(server_addr)
        ) < 0
    ) {
        std::cerr << "bind() failed\n";
        close(server_fd);
        return 1;
    }


    // Listen for incoming connections.
    if (listen(server_fd, 1) < 0) {
        std::cerr << "listen() failed\n";
        close(server_fd);
        return 1;
    }

    std::cout
        << "Server listening on 127.0.0.1:"
        << port
        << '\n';

    // Accept one client connection.
    int client_fd = accept(
        server_fd,
        nullptr,
        nullptr
    );

    if (client_fd < 0) {
        std::cerr << "accept() failed\n";
        close(server_fd);
        return 1;
    }

    std::cout << "Client connected\n";


    // ------------------------------------
    // Receive frame header
    // ------------------------------------

    // Store the payload size from the
    // 4-byte length prefix.
    std::uint32_t payload_size = 0;


    // Receive exactly 4 bytes.
    //
    // recv_exact() expects std::byte*,
    // so reinterpret the address of payload_size.
    bool header_ok = recv_exact(
        client_fd,
        reinterpret_cast<std::byte*>(&payload_size),
        sizeof(payload_size)
    );


    // Check whether the complete header
    // was received successfully.
    if (!header_ok) {
        std::cerr << "Failed to receive frame header\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }

    std::cout
        << "Payload size from header: "
        << payload_size
        << " bytes\n";


    // ------------------------------------
    // Receive payload
    // ------------------------------------

    // Allocate exactly enough space
    // for the payload.
    std::vector<std::byte> payload(
        static_cast<std::size_t>(payload_size)
    );


    // Receive exactly payload_size bytes.
    bool payload_ok = recv_exact(
        client_fd,
        payload.data(),
        payload.size()
    );


    // Check whether the full payload
    // was received successfully.
    if (!payload_ok) {
        std::cerr << "Failed to receive payload\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }

    std::cout
        << "Payload received successfully: "
        << payload.size()
        << " bytes\n";


    // ------------------------------------
    // Deserialize
    // ------------------------------------

    // Convert payload bytes back into ProfileEvent.
    auto event =
        deserializer.deserialize(payload);


    // Check whether deserialization succeeded.
    if (!event.has_value()) {
        std::cerr << "Deserialization failed\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }


    // Inspect and print the reconstructed event.
    std::visit(
        [](const auto& e) {
            using T =
                std::decay_t<decltype(e)>;

            if constexpr (
                std::is_same_v<T, KernelEvent>
            ) {
                std::cout
                    << "Received KernelEvent\n"
                    << "kernel=" << e.kernel_name << '\n'
                    << "timestamp=" << e.timestamp_ns << '\n'
                    << "duration=" << e.duration_ns << '\n'
                    << "device=" << e.device_id << '\n';
            }
            else if constexpr (
                std::is_same_v<T, MemoryEvent>
            ) {
                std::cout
                    << "Received MemoryEvent\n"
                    << "operation=" << e.operation << '\n'
                    << "timestamp=" << e.timestamp_ns << '\n'
                    << "bytes=" << e.bytes << '\n'
                    << "device=" << e.device_id << '\n';
            }
        },
        *event
    );


    // Close connected client socket.
    close(client_fd);


    // Close listening socket.
    close(server_fd);

    return 0;
}

Writing distributed-profiler/apps/reliable_profile_server.cpp


In [34]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/simple_text_deserializer.cpp \
    distributed-profiler/src/tcp_utils.cpp \
    distributed-profiler/apps/reliable_profile_server.cpp \
    -o reliable_profile_server

In [35]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/frame_codec.cpp \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/src/tcp_utils.cpp \
    distributed-profiler/apps/reliable_profile_client.cpp \
    -o reliable_profile_client

In [36]:
!./reliable_profile_server > reliable_server_output.txt 2>&1 & \
SERVER_PID=$!; \
sleep 1; \
./reliable_profile_client; \
wait $SERVER_PID; \
cat reliable_server_output.txt

Payload size: 29 bytes
Frame size: 33 bytes
Frame sent successfully
Server listening on 127.0.0.1:19002
Client connected
Payload size from header: 29 bytes
Payload received successfully: 29 bytes
Received KernelEvent
kernel=gemm_kernel
timestamp=1000
duration=200
device=0


## Step 5 — Send a Batch of Multiple Profiling Events

The previous step validated reliable transmission of one profiling event:

```text
ProfileEvent
    ↓
serialize()
    ↓
frame()
    ↓
send_all()
    ↓
TCP
    ↓
recv_exact()
    ↓
deserialize()
    ↓
ProfileEvent
```

The goal of this step is to send multiple profiling events together in one batch.

---

## Why Batching Is Useful

Without batching, each profiling event may require its own network transmission:

```text
Event 1 → send()
Event 2 → send()
Event 3 → send()
Event 4 → send()
```

This increases:

```text
system call overhead
network overhead
CPU overhead
```

Instead, we can combine multiple framed events into one larger batch:

```text
Event 1 → frame ┐
Event 2 → frame ├→ BatchBuilder → one batch
Event 3 → frame ┤
Event 4 → frame ┘
```

Then:

```text
one batch → send_all()
```

So conceptually:

```text
Without batching:

4 events
→ 4 network sends


With batching:

4 events
→ 1 batch
→ 1 network send
```

---

## Client-Side Data Flow

The client will create multiple events such as:

```text
KernelEvent("gemm_kernel")
MemoryEvent("H2D")
KernelEvent("softmax_kernel")
MemoryEvent("D2H")
```

Each event goes through:

```text
ProfileEvent
    ↓
serialize()
    ↓
payload
    ↓
frame()
    ↓
[length][payload]
```

Then the individual frames are appended into:

```cpp
BatchBuilder
```

Conceptually:

```text
frame 1
frame 2
frame 3
frame 4
    ↓
BatchBuilder
    ↓
[frame1][frame2][frame3][frame4]
```

---

## Batch Layout

Each event frame already contains its own length prefix:

```text
[event_size][event_payload]
```

For example:

```text
[30][KERNEL|1000|200|0|gemm_kernel]
```

A batch containing multiple frames looks like:

```text
[frame1][frame2][frame3][frame4]
```

or more explicitly:

```text
[event1_size][event1_payload]
[event2_size][event2_payload]
[event3_size][event3_payload]
[event4_size][event4_payload]
```

---

## Outer Batch Length Prefix

The server also needs to know how many total bytes belong to the batch.

Therefore, the network protocol should add one more length prefix around the whole batch:

```text
[batch_size][batch_data]
```

The complete layout becomes:

```text
+----------------------+
| batch_size           |
+----------------------+
| event1_size          |
| event1_payload       |
+----------------------+
| event2_size          |
| event2_payload       |
+----------------------+
| event3_size          |
| event3_payload       |
+----------------------+
| event4_size          |
| event4_payload       |
+----------------------+
```

So there are now two framing levels:

```text
Outer framing:
[batch_size][batch]

Inner framing:
[event_size][event_payload]
```

---

## Why Two Levels of Framing Are Needed

The outer length tells the TCP receiver:

```text
How many bytes belong to this batch?
```

The inner lengths tell `BatchParser`:

```text
Where does each individual event begin and end?
```

So:

```text
batch_size
    ↓
defines the network message boundary

event_size
    ↓
defines each event boundary inside the batch
```

---

## Client-Side Pipeline

The client path becomes:

```text
ProfileEvent 1
    ↓
serialize
    ↓
frame
     \
ProfileEvent 2
    ↓
serialize
    ↓
frame
      \
ProfileEvent 3
    ↓
serialize
    ↓
frame
        ↓
    BatchBuilder
        ↓
    batch bytes
        ↓
prepend batch_size
        ↓
    send_all()
        ↓
       TCP
```

---

## Server-Side Pipeline

The server first reads the outer batch header:

```text
recv_exact(4 bytes)
        ↓
batch_size
```

Then it receives exactly the complete batch:

```text
recv_exact(batch_size bytes)
```

Now the server owns:

```text
[frame1][frame2][frame3][frame4]
```

It then passes the batch to:

```cpp
BatchParser
```

which reconstructs:

```text
frame 1
frame 2
frame 3
frame 4
```

Each frame is then processed using:

```text
FrameCodec::unframe()
        ↓
serialized payload
        ↓
SimpleTextDeserializer
        ↓
ProfileEvent
```

---

## Complete End-to-End Flow

```text
Client / Profiler Agent

ProfileEvent
    ↓
Serializer
    ↓
FrameCodec
    ↓
frame
     \
ProfileEvent
    ↓
Serializer
    ↓
FrameCodec
    ↓
frame
      \
       ↓
   BatchBuilder
       ↓
   batch bytes
       ↓
[batch_size][batch]
       ↓
   send_all()
       ↓
========================
       TCP Stream
========================
       ↓
recv_exact(batch header)
       ↓
recv_exact(batch bytes)
       ↓
   BatchParser
       ↓
individual frames
       ↓
 FrameCodec
       ↓
Deserializer
       ↓
ProfileEvent

Server / Collector
```

---

## Example

Suppose the client creates:

```text
Event 1:
KernelEvent
kernel = gemm_kernel

Event 2:
MemoryEvent
operation = H2D

Event 3:
KernelEvent
kernel = softmax_kernel

Event 4:
MemoryEvent
operation = D2H
```

The client generates:

```text
frame1
frame2
frame3
frame4
```

and combines them:

```text
batch =
[frame1][frame2][frame3][frame4]
```

Then sends:

```text
[batch_size][batch]
```

The collector receives the batch and should reconstruct:

```text
KernelEvent
kernel=gemm_kernel

MemoryEvent
operation=H2D

KernelEvent
kernel=softmax_kernel

MemoryEvent
operation=D2H
```

in the same order.

---

## Expected Benefit

Assume:

```text
1000 profiling events
```

Without batching:

```text
1000 events
→ approximately 1000 send operations
```

If the batch size is:

```text
100 events
```

then:

```text
1000 events
→ approximately 10 send operations
```

This reduces syscall overhead substantially.

However, batching introduces a tradeoff:

```text
larger batch
    ↓
fewer network operations
    ↓
better throughput

but

larger batch
    ↓
events wait longer before sending
    ↓
potentially higher latency
```

This throughput-vs-latency tradeoff will become important in the later benchmark.

---

## Files Used in This Step

The client will use:

```text
profile_event.hpp
simple_text_serializer.hpp
frame_codec.hpp
batch_builder.hpp
tcp_utils.hpp
```

The server will use:

```text
profile_event.hpp
simple_text_deserializer.hpp
frame_codec.hpp
batch_parser.hpp
tcp_utils.hpp
```

New application files can be:

```text
apps/batch_profile_client.cpp
apps/batch_profile_server.cpp
```

---

## Validation Criteria

This step is successful if:

```text
1. Client creates multiple ProfileEvents.
2. Every event is serialized correctly.
3. Every event is framed correctly.
4. Frames are appended into one batch.
5. Client sends the batch with send_all().
6. Server receives the complete batch with recv_exact().
7. BatchParser reconstructs all frames.
8. Every frame is successfully unframed.
9. Every payload is successfully deserialized.
10. Reconstructed events match the original events.
11. Event order is preserved.
```

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- Why do we batch profiling events before sending them?
- How does batching reduce system-call overhead?
- What is the difference between an event frame and a batch?
- Why does the batch need an outer length prefix?
- Why do individual events still need their own inner length prefixes?
- How does the server know how many batch bytes to receive?
- How does `BatchParser` recover individual frames?
- How are the original `ProfileEvent` objects reconstructed?
- Why does TCP preserve the event order?
- What is the throughput-versus-latency tradeoff of larger batches?

In [37]:
%%writefile distributed-profiler/apps/batch_profile_client.cpp

#include "batch_builder.hpp"
#include "frame_codec.hpp"
#include "profile_event.hpp"
#include "simple_text_serializer.hpp"
#include "tcp_utils.hpp"

#include <arpa/inet.h>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <sys/socket.h>
#include <unistd.h>
#include <vector>

int main() {
    const int port = 19003;

    SimpleTextSerializer serializer;
    FrameCodec codec;
    BatchBuilder batch_builder;


    // ============================================================
    // Create a TCP socket.
    // ============================================================

    int socket_fd = socket(
        AF_INET,
        SOCK_STREAM,
        0
    );

    if (socket_fd < 0) {
        std::cerr << "socket() failed\n";
        return 1;
    }


    // ============================================================
    // Configure the collector/server address.
    //
    // Target:
    //
    // 127.0.0.1:19003
    // ============================================================

    sockaddr_in server_addr{};

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(port);

    if (
        inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_addr.sin_addr
        ) <= 0
    ) {
        std::cerr << "inet_pton() failed\n";
        close(socket_fd);
        return 1;
    }


    // ============================================================
    // Connect to the collector.
    // ============================================================

    if (
        connect(
            socket_fd,
            reinterpret_cast<sockaddr*>(&server_addr),
            sizeof(server_addr)
        ) < 0
    ) {
        std::cerr << "connect() failed\n";
        close(socket_fd);
        return 1;
    }


    // ============================================================
    // Create several profiling events.
    //
    // These simulate events produced by a GPU profiler.
    // ============================================================

    std::vector<ProfileEvent> events = {
        KernelEvent{
            1000,
            200,
            0,
            "gemm_kernel"
        },

        MemoryEvent{
            1500,
            0,
            4096,
            "H2D"
        },

        KernelEvent{
            2000,
            150,
            0,
            "softmax_kernel"
        },

        MemoryEvent{
            2300,
            0,
            8192,
            "D2H"
        }
    };


    // ============================================================
    // Serialize, frame, and append every event to the batch.
    //
    // Each event goes through:
    //
    // ProfileEvent
    //     ↓
    // serialize()
    //     ↓
    // payload
    //     ↓
    // frame()
    //     ↓
    // [event_size][event_payload]
    //     ↓
    // BatchBuilder
    //
    //
    // Final network message layout:
    //
    // Network message
    // │
    // ├── batch_size
    // │
    // └── batch
    //      │
    //      ├── event1_size
    //      ├── event1_payload
    //      │
    //      ├── event2_size
    //      ├── event2_payload
    //      │
    //      ├── event3_size
    //      ├── event3_payload
    //      │
    //      ├── event4_size
    //      └── event4_payload
    // ============================================================

    for (const auto& event : events) {

        // Convert one ProfileEvent into serialized bytes.
        auto payload =
            serializer.serialize(event);


        // Add the inner event-length prefix:
        //
        // [event_size][event_payload]
        auto frame =
            codec.frame(payload);


        // Append the complete frame into the batch.
        batch_builder.append(frame);
    }


    // ============================================================
    // Obtain the combined batch.
    //
    // Layout:
    //
    // [frame1][frame2][frame3][frame4]
    //
    // Each frame already contains:
    //
    // [event_size][event_payload]
    // ============================================================

    const auto& batch =
        batch_builder.data();


    // ============================================================
    // Create the OUTER batch length.
    //
    // This tells the receiver how many bytes belong to the
    // complete batch.
    //
    // Outer framing:
    //
    // [batch_size][batch]
    //
    // Inner framing:
    //
    // [event_size][event_payload]
    // ============================================================

    std::uint32_t batch_size =
        static_cast<std::uint32_t>(batch.size());


    // ============================================================
    // Send the outer 4-byte batch-size header.
    //
    // send_all() guarantees that all bytes are sent even if
    // one send() call performs only a partial write.
    // ============================================================

    bool header_ok = send_all(
        socket_fd,
        reinterpret_cast<const std::byte*>(&batch_size),
        sizeof(batch_size)
    );

    if (!header_ok) {
        std::cerr
            << "Failed to send batch header\n";

        close(socket_fd);
        return 1;
    }


    // ============================================================
    // Send the complete batch.
    //
    // The batch contains all framed profiling events.
    // ============================================================

    bool batch_ok = send_all(
        socket_fd,
        batch.data(),
        batch.size()
    );

    if (!batch_ok) {
        std::cerr
            << "Failed to send batch data\n";

        close(socket_fd);
        return 1;
    }


    // ============================================================
    // Print useful information.
    // ============================================================

    std::cout
        << "Events sent: "
        << events.size()
        << '\n';

    std::cout
        << "Batch size: "
        << batch.size()
        << " bytes\n";

    std::cout
        << "Batch sent successfully\n";


    // ============================================================
    // Close the client socket.
    // ============================================================

    close(socket_fd);

    return 0;
}

Writing distributed-profiler/apps/batch_profile_client.cpp


In [38]:
%%writefile distributed-profiler/apps/batch_profile_server.cpp

#include "batch_parser.hpp"
#include "frame_codec.hpp"
#include "profile_event.hpp"
#include "simple_text_deserializer.hpp"
#include "tcp_utils.hpp"

#include <arpa/inet.h>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <sys/socket.h>
#include <type_traits>
#include <unistd.h>
#include <variant>
#include <vector>

int main() {
    const int port = 19003;

    BatchParser batch_parser;
    FrameCodec codec;
    SimpleTextDeserializer deserializer;


    // ============================================================
    // Create the server TCP socket.
    // ============================================================

    int server_fd = socket(
        AF_INET,
        SOCK_STREAM,
        0
    );

    if (server_fd < 0) {
        std::cerr << "socket() failed\n";
        return 1;
    }


    // ============================================================
    // Configure server address.
    //
    // Listen on:
    //
    // 127.0.0.1:19003
    // ============================================================

    sockaddr_in server_addr{};

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(port);

    if (
        inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_addr.sin_addr
        ) <= 0
    ) {
        std::cerr << "inet_pton() failed\n";
        close(server_fd);
        return 1;
    }


    // ============================================================
    // Bind the socket to the local address.
    // ============================================================

    if (
        bind(
            server_fd,
            reinterpret_cast<sockaddr*>(&server_addr),
            sizeof(server_addr)
        ) < 0
    ) {
        std::cerr << "bind() failed\n";
        close(server_fd);
        return 1;
    }


    // ============================================================
    // Start listening for incoming TCP connections.
    // ============================================================

    if (listen(server_fd, 1) < 0) {
        std::cerr << "listen() failed\n";
        close(server_fd);
        return 1;
    }

    std::cout
        << "Server listening on 127.0.0.1:"
        << port
        << '\n';


    // ============================================================
    // Accept one profiler-agent connection.
    // ============================================================

    int client_fd = accept(
        server_fd,
        nullptr,
        nullptr
    );

    if (client_fd < 0) {
        std::cerr << "accept() failed\n";
        close(server_fd);
        return 1;
    }

    std::cout << "Client connected\n";


    // ============================================================
    // The incoming network message has two framing levels:
    //
    // Network message
    // │
    // ├── batch_size
    // │
    // └── batch
    //      │
    //      ├── event1_size
    //      ├── event1_payload
    //      │
    //      ├── event2_size
    //      ├── event2_payload
    //      │
    //      ├── event3_size
    //      ├── event3_payload
    //      │
    //      ├── event4_size
    //      └── event4_payload
    //
    //
    // The reconstruction path is:
    //
    // recv batch_size
    //       ↓
    // recv whole batch
    //       ↓
    // BatchParser
    //       ↓
    // frames
    //       ↓
    // FrameCodec::unframe()
    //       ↓
    // payloads
    //       ↓
    // deserialize()
    //       ↓
    // ProfileEvents
    // ============================================================


    // ============================================================
    // Receive the OUTER batch-size header.
    //
    // This tells us exactly how many bytes belong to the batch.
    // ============================================================

    std::uint32_t batch_size = 0;

    bool header_ok = recv_exact(
        client_fd,
        reinterpret_cast<std::byte*>(&batch_size),
        sizeof(batch_size)
    );

    if (!header_ok) {
        std::cerr
            << "Failed to receive batch header\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }

    std::cout
        << "Batch size from header: "
        << batch_size
        << " bytes\n";


    // ============================================================
    // Allocate exactly enough memory for the incoming batch.
    // ============================================================

    std::vector<std::byte> batch(
        static_cast<std::size_t>(batch_size)
    );


    // ============================================================
    // Receive the complete batch.
    //
    // recv_exact() may call recv() multiple times internally
    // until exactly batch_size bytes have been collected.
    // ============================================================

    bool batch_ok = recv_exact(
        client_fd,
        batch.data(),
        batch.size()
    );

    if (!batch_ok) {
        std::cerr
            << "Failed to receive batch data\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }

    std::cout
        << "Batch received successfully: "
        << batch.size()
        << " bytes\n";


    // ============================================================
    // Parse the batch into individual frames.
    //
    // Input:
    //
    // [frame1][frame2][frame3][frame4]
    //
    // Output:
    //
    // frame1
    // frame2
    // frame3
    // frame4
    //
    // BatchParser uses each event's inner length prefix
    // to find where one frame ends and the next begins.
    // ============================================================

    auto frames =
        batch_parser.parse(batch);

    if (!frames.has_value()) {
        std::cerr
            << "Batch parsing failed\n";

        close(client_fd);
        close(server_fd);

        return 1;
    }


    // ============================================================
    // Reconstruct every ProfileEvent.
    //
    // For each frame:
    //
    // [event_size][event_payload]
    //       ↓
    // FrameCodec::unframe()
    //       ↓
    // event_payload
    //       ↓
    // deserialize()
    //       ↓
    // ProfileEvent
    // ============================================================

    std::size_t event_count = 0;

    for (const auto& frame : *frames) {

        // Remove the inner event-length prefix.
        auto payload =
            codec.unframe(frame);

        if (!payload.has_value()) {
            std::cerr
                << "Failed to unframe event\n";

            close(client_fd);
            close(server_fd);

            return 1;
        }


        // Convert serialized payload bytes back into ProfileEvent.
        auto event =
            deserializer.deserialize(*payload);

        if (!event.has_value()) {
            std::cerr
                << "Failed to deserialize event\n";

            close(client_fd);
            close(server_fd);

            return 1;
        }


        // Inspect the concrete event stored inside std::variant.
        std::visit(
            [](const auto& e) {

                using T =
                    std::decay_t<decltype(e)>;


                if constexpr (
                    std::is_same_v<T, KernelEvent>
                ) {
                    std::cout
                        << "KernelEvent\n"
                        << "  kernel="
                        << e.kernel_name
                        << '\n'
                        << "  timestamp="
                        << e.timestamp_ns
                        << '\n'
                        << "  duration="
                        << e.duration_ns
                        << '\n'
                        << "  device="
                        << e.device_id
                        << '\n';
                }

                else if constexpr (
                    std::is_same_v<T, MemoryEvent>
                ) {
                    std::cout
                        << "MemoryEvent\n"
                        << "  operation="
                        << e.operation
                        << '\n'
                        << "  timestamp="
                        << e.timestamp_ns
                        << '\n'
                        << "  bytes="
                        << e.bytes
                        << '\n'
                        << "  device="
                        << e.device_id
                        << '\n';
                }
            },
            *event
        );


        ++event_count;
    }


    // ============================================================
    // Print reconstruction summary.
    // ============================================================

    std::cout
        << "Total reconstructed events: "
        << event_count
        << '\n';


    // ============================================================
    // Close sockets.
    // ============================================================

    close(client_fd);
    close(server_fd);

    return 0;
}

Writing distributed-profiler/apps/batch_profile_server.cpp


In [39]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/batch_parser.cpp \
    distributed-profiler/src/frame_codec.cpp \
    distributed-profiler/src/simple_text_deserializer.cpp \
    distributed-profiler/src/tcp_utils.cpp \
    distributed-profiler/apps/batch_profile_server.cpp \
    -o batch_profile_server

In [40]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/batch_builder.cpp \
    distributed-profiler/src/frame_codec.cpp \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/src/tcp_utils.cpp \
    distributed-profiler/apps/batch_profile_client.cpp \
    -o batch_profile_client

In [41]:
!./batch_profile_server > batch_server_output.txt 2>&1 & \
SERVER_PID=$!; \
sleep 1; \
./batch_profile_client; \
wait $SERVER_PID; \
cat batch_server_output.txt

Events sent: 4
Batch size: 121 bytes
Batch sent successfully
Server listening on 127.0.0.1:19003
Client connected
Batch size from header: 121 bytes
Batch received successfully: 121 bytes
KernelEvent
  kernel=gemm_kernel
  timestamp=1000
  duration=200
  device=0
MemoryEvent
  operation=H2D
  timestamp=1500
  bytes=4096
  device=0
KernelEvent
  kernel=softmax_kernel
  timestamp=2000
  duration=150
  device=0
MemoryEvent
  operation=D2H
  timestamp=2300
  bytes=8192
  device=0
Total reconstructed events: 4
